# Phase 7 - XGBoost Predictive Model
**Problem Statement ID 26184**

Train and evaluate an XGBoost classifier for `future_withdrawal` prediction using chronological splits from Phase 5.

## 1. Phase 7 Objective

In [ ]:
print("PHASE 7: XGBoost model to predict future_withdrawal")
print("Primary metric: PR-AUC | Test set: strictly held out")

## 2. Imports and Paths

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve)
from pathlib import Path
warnings.filterwarnings('ignore')

BASE = Path('../')
PROC = BASE / 'data/processed'
OUT  = BASE / 'outputs'
PRED = OUT / 'predictions'
FIG  = OUT / 'figures'
MDL  = BASE / 'models'
TARGET = 'future_withdrawal'
print(f'XGBoost {xgb.__version__} loaded. Paths configured.')

## 3. Load Datasets

In [ ]:
train_df = pd.read_csv(PROC/'train.csv')
val_df   = pd.read_csv(PROC/'validation.csv')
test_df  = pd.read_csv(PROC/'test.csv')

for name, df in [('TRAIN', train_df), ('VALIDATION', val_df), ('TEST', test_df)]:
    pos = (df[TARGET]==1).sum(); neg = (df[TARGET]==0).sum()
    print(f'{name}: {len(df)} rows | Pos={pos} ({pos/len(df)*100:.2f}%) | Neg={neg}')

## 4. Target Validation

In [ ]:
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    assert TARGET in df.columns
    assert set(df[TARGET].unique()).issubset({0, 1})
dist_df = pd.read_csv(OUT/'phase7_target_distribution.csv')
display(dist_df)
print('Target validation: PASS')

## 5. Safe Feature Selection

In [ ]:
feat_df = pd.read_csv(OUT/'phase7_selected_features.csv')
selected = feat_df[feat_df['selected']==True]['feature'].tolist()
excluded = feat_df[feat_df['selected']==False]
print(f'Selected: {len(selected)} features')
print(f'Excluded: {len(excluded)} columns')
display(excluded[['feature', 'reason']])

## 6. Feature Types

In [ ]:
sub = train_df[selected]
num_cols = sub.select_dtypes(include=['number']).columns.tolist()
cat_cols = sub.select_dtypes(include=['object', 'string']).columns.tolist()
print(f'Numerical : {len(num_cols)}')
print(f'Categorical: {len(cat_cols)}')
print(f'Cat features: {cat_cols}')

## 7. Training-Only Preprocessing

> **CRITICAL:** ColumnTransformer is `fit()` on X_train ONLY. Validation and test use `transform()` only.

In [ ]:
X_train = train_df[selected]; y_train = train_df[TARGET]
X_val   = val_df[selected];   y_val   = val_df[TARGET]
X_test  = test_df[selected]

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    [('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)],
    remainder='drop'
)
X_train_proc = preprocessor.fit_transform(X_train)  # FIT on TRAIN ONLY
X_val_proc   = preprocessor.transform(X_val)         # transform only
print(f'X_train_proc: {X_train_proc.shape}')
print(f'X_val_proc  : {X_val_proc.shape}')
print('Preprocessing fitted on X_train ONLY -- confirmed.')

## 8. Class Imbalance

In [ ]:
neg = int((y_train==0).sum()); pos = int((y_train==1).sum())
spw = neg / pos
print(f'Train negative : {neg:,}')
print(f'Train positive : {pos:,}')
print(f'scale_pos_weight: {neg}/{pos} = {spw:.4f}')

## 9. Initial XGBoost Training

In [ ]:
init_model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='binary:logistic', eval_metric='logloss',
    scale_pos_weight=spw, random_state=42, n_jobs=-1, verbosity=0
)
init_model.fit(
    X_train_proc, y_train,
    eval_set=[(X_train_proc, y_train), (X_val_proc, y_val)],
    verbose=False
)
print('Initial XGBoost trained.')

## 10. Validation Evaluation Helper

In [ ]:
def evaluate(model, X_proc, y_true, threshold=0.50, name='model'):
    proba = model.predict_proba(X_proc)[:, 1]
    preds = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    return {
        'model': name,
        'precision': round(precision_score(y_true, preds, zero_division=0), 4),
        'recall': round(recall_score(y_true, preds, zero_division=0), 4),
        'f1': round(f1_score(y_true, preds, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_true, proba), 4),
        'pr_auc': round(average_precision_score(y_true, proba), 4),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }

init_m = evaluate(init_model, X_val_proc, y_val, name='XGBoost Initial')
print(pd.Series(init_m).to_string())

## 11. Hyperparameter Search Results

In [ ]:
hp_df = pd.read_csv(OUT/'phase7_hyperparameter_results.csv')
display(hp_df.sort_values('pr_auc', ascending=False).reset_index(drop=True))

## 12. Best Model

In [ ]:
best_params = json.load(open(MDL/'phase7_best_parameters.json'))
print('Best configuration:', best_params)
best_model = joblib.load(MDL/'xgboost_best_model.pkl')
best_m = evaluate(best_model, X_val_proc, y_val, name='XGBoost Best')
print(pd.Series(best_m).to_string())

## 13. Comparison with Phase 6 Baselines

In [ ]:
comp_df = pd.read_csv(OUT/'phase7_model_comparison.csv')
display(comp_df)

## 14. Threshold Analysis

In [ ]:
thr_df = pd.read_csv(OUT/'phase7_xgboost_threshold_analysis.csv')
display(thr_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(thr_df['threshold'], thr_df['precision'], 'b-o', label='Precision')
axes[0].plot(thr_df['threshold'], thr_df['recall'],    'r-s', label='Recall')
axes[0].plot(thr_df['threshold'], thr_df['f1'],        'g-^', label='F1')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall / F1 vs Threshold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].bar(thr_df['threshold'].astype(str), thr_df['predicted_positive_count'], color='steelblue')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Predicted Positives')
axes[1].set_title('Predicted Positive Count vs Threshold')
axes[1].grid(axis='y', alpha=0.3)
plt.suptitle('XGBoost Threshold Sensitivity Analysis', fontweight='bold')
plt.tight_layout(); plt.show()
print('Final operational threshold deferred to Phase 8.')

## 15. Feature Importance (Gain)

In [ ]:
fi_df = pd.read_csv(OUT/'xgboost_feature_importance.csv')
top20 = fi_df.head(20).sort_values('importance')
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20['feature'], top20['importance'], color='#2196F3', edgecolor='white')
ax.set_xlabel('Gain Importance'); ax.grid(axis='x', alpha=0.3)
ax.set_title('XGBoost Top-20 Feature Importances (Gain)', fontweight='bold')
plt.tight_layout(); plt.show()
display(fi_df.head(10))

## 16. Training History

In [ ]:
hist_df = pd.read_csv(OUT/'xgboost_training_history.csv')
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_df['iteration'], hist_df['training_logloss'],   label='Train LogLoss', color='#1565C0')
ax.plot(hist_df['iteration'], hist_df['validation_logloss'], label='Val LogLoss',   color='#E53935', linestyle='--')
ax.set_xlabel('Boosting Round'); ax.set_ylabel('LogLoss')
ax.set_title('XGBoost Training History', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 17. ROC Curve Comparison

In [ ]:
img = plt.imread(FIG/'xgboost_vs_baselines_roc.png')
plt.figure(figsize=(9, 7)); plt.imshow(img); plt.axis('off')
plt.title('ROC Curve Comparison -- Phase 7 vs Phase 6 Baselines', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 18. Precision-Recall Comparison

In [ ]:
img = plt.imread(FIG/'xgboost_vs_baselines_pr.png')
plt.figure(figsize=(9, 7)); plt.imshow(img); plt.axis('off')
plt.title('Precision-Recall Comparison -- Phase 7 vs Phase 6', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 19. Leakage Audit

In [ ]:
audit = pd.read_csv(OUT/'phase7_leakage_audit.csv')
display(audit)
fails = audit[audit['status'] == 'FAIL']
assert len(fails) == 0, f'LEAKAGE FAILURES: {fails["check"].tolist()}'
print(f'LEAKAGE AUDIT: {len(audit)}/{len(audit)} checks PASSED.')

## 20. Test Set Protection

In [ ]:
prot = pd.read_csv(OUT/'phase7_test_protection_report.csv')
display(prot)
print()
print('FINAL TEST PERFORMANCE WAS NOT EVALUATED IN PHASE 7.')
print('Test set strictly held out -- deferred to Phase 8.')

## 21. Phase 7 Conclusion

In [ ]:
meta = json.load(open(MDL/'phase7_xgboost_metadata.json'))
vm = meta['best_validation_metrics']
print('=' * 60)
print('PHASE 7 COMPLETE -- XGBOOST PREDICTIVE MODEL')
print('=' * 60)
print(f"  Best Val PR-AUC : {vm['pr_auc']}")
print(f"  Best Val Recall : {vm['recall']}")
print(f"  Best Val F1     : {vm['f1']}")
print(f"  Best Val ROC-AUC: {vm['roc_auc']}")
print(f"  scale_pos_weight: {meta['scale_pos_weight']}")
print(f"  Best Params     : {meta['best_parameters']}")
print(f"  XGBoost Version : {meta['xgboost_version']}")
print()
print('  Test Set: NOT USED FOR MODEL SELECTION OR FINAL EVALUATION')
print('  Leakage : 15/15 PASS')
print('  STATUS  : READY FOR PHASE 8 -- FINAL MODEL EVALUATION')